# OPTUNA + MEDIAN SHAP Số lượng features đầu vào: 133


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import warnings  # 1. Thêm thư viện
import joblib
import json
# 2. Chèn dòng lọc cảnh báo ở đây
warnings.filterwarnings(
    "ignore", message="The reported value is ignored because this `step`.*"
)
from scipy.stats import ks_2samp
from optuna.integration import LightGBMPruningCallback
from optuna.samplers import TPESampler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
app_train_fe = pd.read_parquet("../Data/app_train_fe.parquet")
app_test_fe = pd.read_parquet("../Data/app_test_fe.parquet")
mean_shap = pd.read_csv("../Data/mean_shap.csv")

In [ ]:
# Tách features và target
X_train = app_train_fe.drop(columns=["TARGET"])
X_test = app_test_fe.copy()
y_train = app_train_fe["TARGET"]

In [ ]:
# Loại bỏ SK_ID_CURR vì chỉ là ID định danh, không có giá trị dự báo
# Giữ lại để dùng sau (submit Kaggle, OOF predictions)
features = [col for col in X_train.columns if col not in ["SK_ID_CURR", "TARGET"]]
print(f"Số features đưa vào model: {len(features)}")

# Lấy danh sách cột category để khai báo với LightGBM
# LightGBM xử lý category trực tiếp mà không cần One-Hot hay Label Encoding
cat_cols = X_train[features].select_dtypes(include="category").columns.tolist()
print(f"Số cột category: {len(cat_cols)}")

In [ ]:
# ==============================================================================
# LỌC FEATURES THEO NGƯỠNG MEAN SHAP (GIỮ LẠI 66 FEATURES)
# ==============================================================================

# Lấy ngưỡng = mean của mean_abs_shap
shap_threshold_mean = mean_shap["mean_abs_shap"].mean()
print(f"Ngưỡng SHAP (Mean): {shap_threshold_mean:.6f}")

# Lấy danh sách features có mean |SHAP| >= ngưỡng mean
selected_features_mean = mean_shap[mean_shap["mean_abs_shap"] >= shap_threshold_mean][
    "feature"
].tolist()

print(f"Số features ban đầu: {len(features)}")
print(f"Số features sau lọc (Mean): {len(selected_features_mean)}")
print(f"Số features bị loại (Mean): {len(features) - len(selected_features_mean)}")

# Cập nhật cat_cols cho bộ features lọc theo Mean
cat_cols_filtered_mean = [col for col in cat_cols if col in selected_features_mean]
print(f"Số cột category sau lọc (Mean): {len(cat_cols_filtered_mean)}")
print("-" * 50)


# ==============================================================================
# BỔ SUNG: LỌC FEATURES THEO NGƯỠNG MEDIAN SHAP (TRUNG VỊ)
# ==============================================================================

# Lấy ngưỡng = median của mean_abs_shap
# Thích hợp khi dữ liệu SHAP có một vài feature quá vượt trội (outliers) làm lệch giá trị Mean
shap_threshold_median = mean_shap["mean_abs_shap"].median()
print(f"Ngưỡng SHAP (Median): {shap_threshold_median:.6f}")

# Lấy danh sách features có mean |SHAP| >= ngưỡng median
selected_features_median = mean_shap[
    mean_shap["mean_abs_shap"] >= shap_threshold_median
]["feature"].tolist()

print(f"Số features sau lọc (Median): {len(selected_features_median)}")
print(f"Số features bị loại (Median): {len(features) - len(selected_features_median)}")
# Cập nhật cat_cols cho bộ features mới theo Median
cat_cols_filtered_median = [col for col in cat_cols if col in selected_features_median]
print(f"\nSố cột category sau lọc (Median): {len(cat_cols_filtered_median)}")

In [ ]:
# 1. Khởi tạo bản sao của X_test
test_processed_df = X_test.copy()

# 2. Lọc DataFrame của test_processed_df theo selected_features_median
test_processed_df = test_processed_df[list(selected_features_median)]

# 3. Kiểm tra và xử lý cột định danh SK_ID_CURR
# Nếu cột SK_ID_CURR chưa nằm trong danh sách tính năng, chúng ta sẽ lấy lại từ X_test gốc
if "SK_ID_CURR" not in test_processed_df.columns:
    # Trường hợp SK_ID_CURR đang là cột trong X_test
    if "SK_ID_CURR" in X_test.columns:
        test_processed_df.insert(0, "SK_ID_CURR", X_test["SK_ID_CURR"].values)
    # Trường hợp bạn đã đặt SK_ID_CURR làm Index của DataFrame trước đó
    else:
        test_processed_df.insert(0, "SK_ID_CURR", X_test.index)
else:
    # Nếu SK_ID_CURR đã có sẵn, chỉ cần đưa nó lên vị trí cột đầu tiên cho đúng cấu trúc
    cols = ["SK_ID_CURR"] + [c for c in test_processed_df.columns if c != "SK_ID_CURR"]
    test_processed_df = test_processed_df[cols]

# 4. Xuất ma trận dữ liệu tập Test sạch ra file CSV phục vụ cho giao diện Batch Prediction
# Giữ nguyên các ô trống (NaN của Pandas), không cần biến đổi thành None ở bước này
output_path = "../Data/artifacts/test_processed.csv"
test_processed_df.to_csv(output_path, index=False)

print("--- HOÀN THÀNH XUẤT DỮ LIỆU TEST ---")
print(f"-> Đường dẫn file: {output_path}")
print(
    f"-> Cấu trúc ma trận: {test_processed_df.shape} (Bao gồm 1 cột ID + các biến SHAP)"
)

In [ ]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "random_state": 42,
        # Giới hạn số core của mỗi model để nhường core cho Optuna chạy song song các trial
        "n_jobs": 2,
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        # Giữ một đại diện cho việc subsample feature
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        # Giữ một đại diện cho subsample data dòng + kèm freq để kích hoạt
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_estimators": 2000,
    }

    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in skf_tune.split(
        X_train[selected_features_median], y_train
    ):
        X_tr = X_train[selected_features_median].iloc[train_idx]
        X_val = X_train[selected_features_median].iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                # Callback cắt tỉa: Nếu fold hiện tại học quá tệ so với lịch sử, dừng trial ngay
                LightGBMPruningCallback(trial, "auc"),
            ],
            categorical_feature=cat_cols_filtered_median,
        )

        val_preds = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, val_preds))

    return np.mean(auc_scores)


# Khởi tạo study tích hợp thuật toán cắt tỉa MedianPruner
# Thử nghiệm sẽ bị xét cắt tỉa sau vòng lặp (cây thứ) 15 nếu không có tiềm năng
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=15),
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(
    f"Bắt đầu Optuna tuning tối ưu trên {len(selected_features_median)} features (median SHAP) ..."
)
print("=" * 50)

# Chạy song song 2 trial cùng lúc nhờ n_jobs=2
study.optimize(
    objective, n_trials=50, show_progress_bar=True, n_jobs=2, gc_after_trial=True
)

print(f"\nBest AUC (3-fold): {study.best_value:.4f}")
print(f"Best GINI (3-fold): {study.best_value * 2 - 1:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Các điểm cần sửa đổi trong Code của bạn:
Tăng n_warmup_steps lên tối thiểu 50 hoặc 100: Cho phép các mô hình có tốc độ học thấp có đủ thời gian để thể hiện sức mạnh trước khi bị xét cắt tỉa.

Hạ trần num_leaves xuống mức tối đa 64 hoặc 128: Với cấu trúc dữ liệu phẳng (tabular data), num_leaves: 207 là quá lớn và gây nhiễu nghiêm trọng.

In [ ]:
# Ở cây thứ 15, kết quả cho ra kém hơn baseline filtered median
# vì vậy tăng lên 60, hủy trễ hơn để model học chậm còn thời gian học


def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "random_state": 42,
        # Giới hạn số core của mỗi model để nhường core cho Optuna chạy song song các trial
        "n_jobs": 2,
        "num_leaves": trial.suggest_int("num_leaves", 20, 128),  # thay đổi lá
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),  # thay đổi ngưỡng học
        # Giữ một đại diện cho việc subsample feature
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        # Giữ một đại diện cho subsample data dòng + kèm freq để kích hoạt
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_estimators": 2000,
    }

    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in skf_tune.split(
        X_train[selected_features_median], y_train
    ):
        X_tr = X_train[selected_features_median].iloc[train_idx]
        X_val = X_train[selected_features_median].iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                # Callback cắt tỉa: Nếu fold hiện tại học quá tệ so với lịch sử, dừng trial ngay
                LightGBMPruningCallback(trial, "auc"),
            ],
            categorical_feature=cat_cols_filtered_median,
        )

        val_preds = model.predict_proba(X_val)[:, 1] # Xác suất vỡ nợ trên tập val
        auc_scores.append(roc_auc_score(y_val, val_preds))

    return np.mean(auc_scores)


# Khởi tạo study tích hợp thuật toán cắt tỉa MedianPruner
# Thử nghiệm sẽ bị xét cắt tỉa sau vòng lặp (cây thứ) 60 nếu không có tiềm năng
# Ở cây thứ 15, kết quả cho ra kém hơn baseline filtered median vì vậy tăng lên 60
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=60),
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(
    f"Bắt đầu Optuna tuning tối ưu trên {len(selected_features_median)} features (median SHAP) ..."
)
print("=" * 50)

# Chạy song song 2 trial cùng lúc nhờ n_jobs=2
study.optimize(
    objective, n_trials=50, show_progress_bar=True, n_jobs=2, gc_after_trial=True
)

print(f"\nBest AUC (3-fold): {study.best_value:.4f}")
print(f"Best GINI (3-fold): {study.best_value * 2 - 1:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [ ]:
# 1. Khai báo bộ tham số tốt nhất thu được từ Optuna
# Lưu ý: Đổi n_jobs thành -1 để tận dụng tối đa tất cả các nhân CPU khi chạy đơn lẻ
best_params = {
    "objective": "binary",
    "metric": "auc",
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1,
    "num_leaves": 98,
    "learning_rate": 0.015829767735648315,
    "colsample_bytree": 0.6021202840501053,
    "subsample": 0.8649562804399532,
    "bagging_freq": 1,
    "min_child_samples": 231,
    "reg_alpha": 0.05997894005023057,
    "reg_lambda": 0.041170214980396795,
    "n_estimators": 2000,
}

# 2. Khởi tạo K-Fold với 5 splits (giống hệt cấu hình mốc baseline)
skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Khởi tạo mảng chứa dự đoán Out-of-Fold (OOF) cho toàn bộ tập dữ liệu
oof_preds = np.zeros(len(X_train))
auc_scores = []
models_trained = []

print("============================================================")
# Đoạn text tự động cập nhật số lượng feature thực tế (133)
print(
    f" ĐÁNH GIÁ CHIẾN LƯỢC TỐI ƯU HÓA: {len(selected_features_median)} FEATURES (MEDIAN SHAP)"
)
print("============================================================")

# Vòng lặp huấn luyện qua 5 Fold và lưu lại để chuẩn bị sử dụng lại
for fold, (train_idx, val_idx) in enumerate(
    skf_final.split(X_train[selected_features_median], y_train), 1
):

    # Chia dữ liệu theo index của fold hiện tại
    X_tr = X_train[selected_features_median].iloc[train_idx]
    X_val = X_train[selected_features_median].iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Khởi tạo mô hình đơn lẻ
    model = lgb.LGBMClassifier(**best_params)

    # Huấn luyện mô hình (Không dùng callback cắt tỉa của Optuna nữa)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        categorical_feature=cat_cols_filtered_median,
    )

    # Dự đoán xác suất cho tập validation của fold này và lưu vào mảng OOF
    val_preds = model.predict_proba(X_val)[:, 1]  # Xác suất vỡ nợ trên tập val
    oof_preds[val_idx] = val_preds  # Gán vào đúng vị trí trong mảng 300k dòng dữ liệu

    # Tính toán chỉ số đánh giá cho riêng fold này
    fold_auc = roc_auc_score(y_val, val_preds)
    fold_gini = fold_auc * 2 - 1
    auc_scores.append(fold_auc)
    
    print(
        f"Fold {fold}/5...  Fold {fold} — AUC: {fold_auc:.4f} | GINI: {fold_gini:.4f} | Best iteration: {model.best_iteration_}"
    )
    model_filename = f"../Data/artifacts/lgb_fold_{fold}.pkl"
    joblib.dump(model, model_filename)
    models_trained.append(model_filename)
    print(f"Đã đóng gói và lưu mô hình fold {fold} thành file {model_filename}\n")

# 3. Tính toán điểm đánh giá OOF cuối cùng trên toàn bộ tập dữ liệu
oof_auc_final = roc_auc_score(y_train, oof_preds)  # so sánh dự đoán với kết quả thật
oof_gini_final = oof_auc_final * 2 - 1
# Tách xác suất theo nhóm
preds_positive = oof_preds[y_train == 1]  # người vỡ nợ
preds_negative = oof_preds[y_train == 0]  # người không vỡ nợ

ks_stat, _ = ks_2samp(preds_positive, preds_negative)
print(f"KS Statistic: {ks_stat:.4f}")

print("\n" + "=" * 65)
print("                BẢNG ĐỐI CHIẾU HIỆU NĂNG SAU TỐI ƯU                ")
print("=" * 65)
print(
    f"Mốc 3 (Median SHAP - Đã tuned) — Filtered ({len(selected_features_median)} features):"
)
print(f"  OOF AUC:  {oof_auc_final:.4f} | OOF GINI: {oof_gini_final:.4f}")
print(f"  Trung bình AUC giữa các fold: {np.mean(auc_scores):.4f}")
print("=" * 65)

In [ ]:
# 1. Xuất cấu trúc biến để Backend đối chiếu thứ tự và ép kiểu dữ liệu
metadata = {
    "feature_names": list(selected_features_median),
    "categorical_features": list(cat_cols_filtered_median),
}

with open("../Data/artifacts/model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)
print("Đã tạo xong: model_metadata.json")

# 2. Trích xuất 5 dòng đầu tiên từ tập train làm dữ liệu mẫu (Mock data) để test API công thức JSON
mock_df = X_train[selected_features_median].head(5).copy()
# 3. Lấy bổ sung TARGET tương ứng 5 dòng vừa lấy
mock_df["TARGET"] = y_train.head(5).values
# 4. Ép NaN thành None
mock_df = mock_df.replace({np.nan: None})
# 5. Chuyển DataFrame thành Dict
mock_samples = mock_df.to_dict(orient="records")
# 6. Xuất danh sách thành json
with open("../Data/artifacts/mock_samples.json", "w", encoding="utf-8") as f:
    json.dump(mock_samples, f, ensure_ascii=False, indent=4)
print("Đã tạo xong: mock_samples.json (Chứa 5 hồ sơ khách hàng mẫu)")